In [ ]:
import torch
import numpy as np
import pandas as pd
import os
import json 
from os.path import dirname


root_path = dirname(os.getcwd()) + "/SephigraphV2"

pd.set_option("display.max_columns", None)
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs/"

print(root_path, data_dir_processed, data_dir_graphs, sep="\n")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
PATIENCE = 10
NUM_EPOCHS = 200
TOT_TRIALS = 20

In [ ]:
with open("data/dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [ ]:
list(datasets_info.keys())

In [ ]:
dataset = "BPI20_RequestForPayment"

In [ ]:
with open("data/dataset_features.json", 'r') as file:
    dataset_info = json.load(file)[dataset]
dataset_info

In [ ]:
tab_all = pd.read_csv(f"{data_dir_processed}{dataset}/{dataset}_processed_all.csv") 

In [ ]:
print(f"number of events {len(tab_all)}")
print(f"number of traces {len(tab_all['CaseID'].unique())}")
print(f"number of attributes {len(tab_all.columns) - 1}") 

In [ ]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

In [ ]:
list_unique = {k : list(tab_all[k].unique()) for k in categorical_columns}

In [ ]:
import random


torch.manual_seed(0)
torch.cuda.manual_seed(0)
random.seed(0)
np.random.seed(0)

In [ ]:
data_dir_graphs

In [ ]:
from torch_geometric.data.hetero_data import HeteroData

torch.serialization.add_safe_globals([HeteroData])

In [ ]:
X_TRAIN = torch.load(f"{data_dir_graphs}{dataset}/train_set.pt", weights_only=False)
X_VALID = torch.load(f"{data_dir_graphs}{dataset}/validation_set.pt", weights_only=False)
X_TEST = torch.load(f"{data_dir_graphs}{dataset}/test_set.pt", weights_only=False)

In [ ]:
from torch_geometric.loader import DataLoader

In [ ]:
edge_types = set()
node_types = set()
for i in range(len(X_TRAIN)):
    n, edge_type = X_TRAIN[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_VALID)):
    n, edge_type = X_VALID[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_TEST)):
    n, edge_type = X_TEST[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)

NODE_TYPES = list(node_types)
EDGE_TYPES = list(edge_types)

In [ ]:
NODE_TYPES

In [ ]:
EDGE_TYPES

## Hyperopt

In [ ]:
from ax.service.managed_loop import optimize
from torch_geometric.nn import (
    HeteroConv,
    GATv2Conv,global_mean_pool
)
from torch.nn import (
    ModuleList,
    Module,
    Linear,
    ModuleDict,
  )
from typing_extensions import Self

In [ ]:
class HGNN(Module):

    def __init__(self, output_cat, output_real, nodes_relations, parameters) -> Self:
        super().__init__()

        # List of convolutional layers

        hid = parameters["hid"]
        layers = parameters["layers"]
        aggregation = parameters["aggregation"]

        self.output_cat = output_cat
        self.output_real = output_real

        self.convs = ModuleList()
        for _ in range(layers):
            conv = HeteroConv(
                {
                    relation: (
                        GATv2Conv(
                            (-1, -1), hid, 1, False, add_self_loops=False, residual=False
                        )
                    )
                    for relation in nodes_relations
                },
                aggr=aggregation,
            )

            self.convs.append(conv)

        self.FC = ModuleDict()

        for k in output_cat:
            self.FC[k] = Linear(hid, output_cat[k])
        for k in output_real:
            self.FC[k] = Linear(hid, 1)

    def forward(self, batch):

        for i in range(len(self.convs)):
            batch.x_dict = self.convs[i](
                batch.x_dict, batch.edge_index_dict
            )

            batch.x_dict = {key: x.relu() for key, x in batch.x_dict.items()}

        output = {}

        for k in self.output_cat:
            output[k] = global_mean_pool(batch.x_dict[k], batch[k].batch)
            output[k] = self.FC[k](output[k])
        for k in self.output_real:
            output[k] = global_mean_pool(batch.x_dict[k], batch[k].batch)
            output[k] = self.FC[k](output[k]).reshape(1, -1)[0]

        return output

In [ ]:
from torcheval.metrics.functional import multiclass_accuracy  
import torch.nn as nn
from copy import deepcopy
from tqdm.notebook import tqdm
from time import time

from torch.nn.functional import l1_loss
from torcheval.metrics.functional import multiclass_f1_score 

In [ ]:
def train_hgnn(config, output_cat, output_real):
    print(config)

    net = HGNN(
        parameters=config,
        output_cat=output_cat,
        output_real=output_real,
        nodes_relations=EDGE_TYPES,
    )

    net = net.to(device)

    losses = {}

    for k in output_cat:
        losses[k] = nn.CrossEntropyLoss()
    for k in output_real:
        losses[k] = nn.L1Loss()

    
    train_loader = DataLoader(X_TRAIN, batch_size=config["batch_size"], shuffle=True)
    valid_loader = DataLoader(X_VALID, batch_size=config["batch_size"], shuffle=True)
    
    optimizer = torch.optim.Adam(
        net.parameters(), lr=config["lr"]
    )

    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,mode="min", patience=int(PATIENCE/2)
    )
    
    
    best_model = None
    best_loss = 0
    best_f1 = 0
    patience = PATIENCE
    pat_count = 0

    torch.cuda.empty_cache()

    training_times = []
    
    
    
    for epoch in tqdm(range(0, NUM_EPOCHS)):

        net.train()

        

        for _, x in enumerate(train_loader):
            x = x.to(device)
            
            labels = x.y

            optimizer.zero_grad()
            
            outputs = net(x)

            losses_step = {k: losses[k](outputs[k], labels[k]) for k in losses}

            total_loss = 0
            for k in losses_step:
                total_loss += losses_step[k]

            total_loss.backward()
            optimizer.step()

        
        running_total_loss = []
        
        
        
        predictions_categorical = {k: [] for k in output_cat}
        target_categorical = {k: [] for k in output_cat}
        avg_MAE = {k : [] for k in output_real}
        prediction_numerical = {k: [] for k in output_real}
        target_numerical = {k: [] for k in output_real}
        
        net.eval()
        with torch.no_grad():
            for i, x in enumerate(valid_loader):
                x = x.to(device)
            
                labels = x.y

                optimizer.zero_grad()
                
                outputs = net(x)

                losses_step = {k: losses[k](outputs[k], labels[k]) for k in losses}

                running_total_loss.append(sum(list(losses_step.values())))
            
            for k in output_cat:
                predictions_categorical[k].append(
                        torch.argmax(torch.softmax(outputs[k], dim=1), 1)
                )
                target_categorical[k].append(labels[k])
            
            
            for k in output_real:
                prediction_numerical[k].append(
                    outputs[k]
                )
                target_numerical[k].append(
                    labels[k]
                )

        
        val_loss = sum(running_total_loss) / len(running_total_loss)
        
        lr_scheduler.step(val_loss)
        
        for k in predictions_categorical:
            predictions_categorical[k] = torch.cat(predictions_categorical[k])
            target_categorical[k] = torch.cat(target_categorical[k])
                
        for k in prediction_numerical:
            prediction_numerical[k] = torch.cat(prediction_numerical[k])
            target_numerical[k] = torch.cat(target_numerical[k])
    
        
       
        
        
        macro_f1s = {
            k : multiclass_f1_score(
                predictions_categorical[k],
                target_categorical[k].squeeze(),
                num_classes=output_cat[k],
                average='macro'
            ) 
            for k in output_cat
        }
        
        accuracy = {
                k: multiclass_accuracy(
                    predictions_categorical[k],
                    target_categorical[k].squeeze(),
                    num_classes=output_cat[k],
                )
                for k in output_cat
            }
        
        MAE = {
            k: l1_loss(prediction_numerical[k], target_numerical[k]).item()
            for k in output_real
        }

        
        
        f1_activity = macro_f1s["Activity"]
        
        
    
        if epoch == 0:
            best_model = deepcopy(net)
            best_loss = val_loss
            best_f1 = f1_activity
            print("/"*10)
            print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
            print("Acc", accuracy)
            print("F1s", macro_f1s)
            print("MAE", MAE)
            print(f"Patience {pat_count}/{patience}, val loss {val_loss} current_lr {lr_scheduler.get_last_lr()}, curr_best_activity_F1 {best_f1}")
        else:
            
            if val_loss < best_loss:
                best_loss = val_loss
                best_model = deepcopy(net)
                pat_count = 0
            if best_f1 < f1_activity:
                best_f1 = f1_activity

            print("/"*10)
            print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
            print("Acc", accuracy)
            print("F1s", macro_f1s)
            print("MAE", MAE)
            print(f"Patience {pat_count}/{patience}, val loss {val_loss} current_lr {lr_scheduler.get_last_lr()}, curr_best_activity_F1 {best_f1}")
            if pat_count == patience:
                return ({"valid_loss" : best_loss.item()} , best_model)
        pat_count += 1
    print("Acc", accuracy)
    print("F1s", macro_f1s)
    print("MAE", MAE)
    print(f"Patience {pat_count}/{patience}, val loss {val_loss} current_lr {lr_scheduler.get_last_lr()}, curr_best_activity_F1 {best_f1}")
    return ({"valid_loss" : best_loss.item()} , best_model)
   

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
def test_hgnn(net, output_cat, output_real):

    losses = {}

    for k in output_cat:
        losses[k] = nn.CrossEntropyLoss()
    for k in output_real:
        losses[k] = nn.L1Loss()

    predictions_categorical = {k: [] for k in output_cat}
    target_categorical = {k: [] for k in output_cat}

    avg_MAE = {k: [] for k in output_real}
    prediction_numerical = {k: [] for k in output_real}
    target_numerical = {k: [] for k in output_real}

    running_total_loss = []

    test_loader = DataLoader(X_TEST, 128, shuffle=False)
    
    net.eval()
    with torch.no_grad():
        for i, x in enumerate(test_loader):
            x = x.to(device)
        
            labels = x.y
            outputs = net(x)
            losses_step = {k: losses[k](outputs[k], labels[k]) for k in losses}
            running_total_loss.append(sum(list(losses_step.values())))
        
        for k in output_cat:
            predictions_categorical[k].append(
                    torch.argmax(torch.softmax(outputs[k], dim=1), 1)
            )
            target_categorical[k].append(labels[k])
        
        
        for k in output_real:
            prediction_numerical[k].append(
                outputs[k]
            )
            target_numerical[k].append(
                labels[k]
            )
    
    for k in predictions_categorical:
        predictions_categorical[k] = torch.cat(predictions_categorical[k])
        target_categorical[k] = torch.cat(target_categorical[k])
            
    for k in prediction_numerical:
        prediction_numerical[k] = torch.cat(prediction_numerical[k])
        target_numerical[k] = torch.cat(target_numerical[k])
    
    macro_f1s = {
        k : multiclass_f1_score(
            predictions_categorical[k],
            target_categorical[k].squeeze(),
            num_classes=output_cat[k],
            average='macro'
        )
        for k in output_cat
    }
        
    accuracy = {
            k: multiclass_accuracy(
                predictions_categorical[k],
                target_categorical[k].squeeze(),
                num_classes=output_cat[k],
            )
            for k in output_cat
        }
        
    MAE = {
        k: l1_loss(prediction_numerical[k], target_numerical[k]).item()
        for k in output_real
    }
    

    avg_MAE = {k: sum(avg_MAE[k]) / len(avg_MAE[k]) for k in avg_MAE}


    Average_total_loss = sum(running_total_loss) / len(running_total_loss)

    res = (
        {f"{k}_acc": accuracy[k].item() for k in accuracy}
        | {f"{k}_macroF1": macro_f1s[k].item() for k in macro_f1s}
        | {f"{k}_mae": avg_MAE[k] for k in avg_MAE}
        | {"AVG_total_loss": Average_total_loss.item()}
        | {f"MAE_{k}": MAE[k] for k in MAE}
    )

    print(res)

    return res

In [ ]:

outputcat = {"Activity" : len(list_unique["Activity"])}
outputreal = []


print(outputcat)
print(outputreal)

In [ ]:
import logging

logging.getLogger("root").setLevel(logging.ERROR)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
def train_evaluate(config):
    res, _ = train_hgnn(config, output_cat=outputcat, output_real=outputreal)
    return res

In [ ]:
best_parameters, values, experiment, model = optimize(
    parameters=[
        {"name": "hid", "type": "choice", "values": [128], "value_type": "int", "is_ordered" : True,"sort_values":False},
        {"name": "layers", "type": "choice", "values": [2,4], "value_type": "int", "is_ordered" : True, "sort_values":False},
        {"name": "lr", "type": "range", "bounds": [1e-4, 1e-2], "value_type": "float", "log_scale": True},
        {"name": "aggregation", "type" : "choice", "values" :["sum", "mean", "max"], "value_type" : "str"},
        {"name": "batch_size", "type": "choice", "values": [16,64,128,256,512], "value_type": "int", "is_ordered" : True,"sort_values":False},
    ],
  
    evaluation_function=train_evaluate,
    objective_name='valid_loss',
    arms_per_trial=1,
    minimize = True,
    random_seed = 123,
    total_trials = TOT_TRIALS
)

print(best_parameters)
means, covariances = values
print(means)
print(experiment)

In [ ]:
from ax.service.utils.report_utils import exp_to_df

results = exp_to_df(experiment)

In [ ]:
results.sort_values(by="valid_loss")

In [ ]:
results = results.sort_values(by="valid_loss")

In [ ]:
if not os.path.isdir(f"results/{dataset}"):
    os.mkdir(f"results/{dataset}")

In [ ]:
results.to_csv(f"results/{dataset}/hyp_params_search.csv", sep=",", index=False)

## Test on 10 models

In [ ]:
import pandas


def create_df(results):
    res = {}
    
    for k in results[0]:
        res[k] = [x[k] for x in results]
    print(res)
    breakpoint()
    res = pandas.DataFrame(data=res)
    
    return res, res.mean(), res.std()
        

In [ ]:
def test_multi(config, outputcat, outputreal, num_runs=10):
    
    res = []
    
    save_path = f"results/{dataset}/"
    
    for i in range(num_runs):
        
        print(f"Run {i}")
        
        _,net = train_hgnn(
                config, 
                outputcat,
                outputreal,
            )
        
        res.append(
            test_hgnn(
                net,
                outputcat,
                outputreal,
            )
        )
        
        print("RES:")
        print(res[-1])
    
    
    results_table, means, stds = create_df(res)
                
    results_table.to_csv(f"{save_path}results.csv", sep=",", index=False)
        
    pd.DataFrame(data={"mean" : means, "std" : stds}).to_csv(f"{save_path}mean_and_stds.csv", sep=",")
        
    print(pd.DataFrame(data={"mean" : means, "std" : stds}))
        
        
    
    return res

In [ ]:
test_multi(best_parameters, outputcat, outputreal, 10)